<H1>Exponential Families</H1>

In [ ]:
from __future__ import annotations
import abc
import functools

from jax import numpy as jnp
from numpy.typing import ArrayLike

$\large\text{Definition: Exponential Family}$<br>
$\large\text{Consider a random variable }X\text{ taking values }x\in\mathbb{X}\subset\mathbb{R}^{N}.\text{ A probability distribution for }X\text{ with pdf of the functional form}$<br>
$\large p_{w}\left(x\right)=\text{exp}\left[\phi\left(x\right)^{T}w-g\left(w\right)+h\left(x\right)\right]=\frac{H\left(x\right)}{Z\left(w\right)}e^{\phi\left(x\right)^{T}w}=p(x|w)$<br>
$\large\text{is called an exponential family of probability measures.}$<br>
$\large\quad\text{The function }\phi:\mathbb{X}\rightarrow\mathbb{R}^{d}\text{ is called the sufficient statistics}$<br>
$\large\quad\text{The parameters }w\in\mathbb{R}^{d}\text{ are the natural parameters of}p_{w}$<br>
$\large\quad\text{The normalization constant }Z(w):\mathbb{R}^{d}\rightarrow\mathbb{R}\text{ is the partition function: }Z(w)=\int e^{\phi(x)^{T}w+h(x)}dx\text{ and}$<br>
$\large\quad g(w)=log(Z(w)\text{ is the log-partition function}$<br>
$\large\quad \text{The function }h(x):\mathbb{X}\rightarrow\mathbb{R}_{+}\text{ is the carrier measure, with }H(x)=exp\left[h(x)\right]$<br>
$\text{The parameterization is not unique. Any bijective linear mapping }w'=Aw,\phi'(x)=A^{-T}\phi\text{ gives the same distribution. More generally, }w\text{ can be replaced by some arbitraray function }w(\theta)\text{ of paramters }\theta.$<br>

$\large\text{Definition: Conjugate Prior}$<br>
$\large\text{Let }D\text{ and }x\text{ be a data-set and a variable to be inferred, respectively, connected by the likelihood }p(D|x)=\mathcal{l}(D;x).\text{ A conjugate prior to }\mathcal{l}\text{ for }x\text{ is a probability measure with pdf }p(x)=f(x;\theta)\text{ such that}$<br>
$\large\frac{\mathcal{l}(D;x)f(x;\theta)}{\int\mathcal{l}(D;x)f(x;\theta)dx}=f(x;\theta+\phi(D))$<br>
$\large\text{That is, such that the posterior arising from }\mathcal{l}\text{ is of the same functional form as the prior, with updated parameters arising by adding some sufficient statistics of the obseration }D\text{ to the prior's paramerters.}$<br>
$\large\text{Think of }f\text{ as a class with constructor }\theta\text{ and update method }\phi.$<br>

$\large\text{Every exponential family has a congugate prior (up to normalization).}$[[1]](#References)<br>
$\large\text{Consider the exponential family}\quad\quad\quad\quad p(x|w)=exp\left[\phi(x)^{T}w-g(w)+h(x)\right]$<br>
$\large\text{Then a conjugate prior is}\quad\quad\quad\quad p(w|\alpha,\nu)=exp\begin{bmatrix}\begin{pmatrix}w\\-g(w)\end{pmatrix}^{T}&\begin{pmatrix}\alpha&\nu\end{pmatrix}-f(\alpha,\nu)\end{bmatrix}$<br>
$\large\text{with}\quad\quad\quad\quad\quad\quad\quad\quad f(\alpha,\nu):=log\int exp[\alpha^{T}w-\nu g(w)]dw$<br>
$\large\text{Because}\quad\quad\quad\quad\quad\quad p(w|x)\propto p(x|w)p(w)\propto p(w|\alpha+\phi(x),\nu+1)$<br>
$\large\text{and the predictive is}\quad\quad\quad\quad\quad\quad p(x_{+})=\int p(x_{+}|w)p(w)dw$<br>
$\large\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad=exp(h(x_{+}))\int exp[(\phi(x_{+}+\alpha)^{T}w-(\nu+1)g(w)-f(\alpha,nu)]dw$<br>
$\large\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad\quad=exp(h(x_{+}))+f(\alpha+\phi(x_{+},\nu+1)-f(\alpha,\nu)]$<br>
$\large\text{The log-partition function }f(\alpha,\nu)\text{ is the enabling ingredient for full Bayesian infernce on }p(x|w)\text{ (including the marginal on } x \text{ and the posterior on }w\text{). The function }f\text{ maps inference to addition.}$<br>


In [ ]:
class ExponentialFamily(abc.ABC):
    @abc.abstractmethod
    def sufficient_statistics(self x: ArrayLike | jnp.ndarray, \) -> jnp.ndarray:
        """Signature '(D)->({)'"""
    @abc.abstractmethod
    def log_base_measure(self, x: ArrayLike | jnp.ndarray, \) -> jnp.ndarray:
        """Signature '(D)->()'"""
    @abc.abstractmethod
    def log_partition(self, natural_paramaters: ArrayLike | jnp.ndarray, \) -> jnp.ndarray:
        """Signature '(P)->()'"""

    def logpdf(self, x: ArrayLike | jnp.ndarray, natural_parameters: ArrayLike | jnp.ndarray, \) -> jnp.ndarray: # '(D), (P)->()'
        # log p(x|w) = log h(x) + sufficient_statistics(x) @ w - log Z(w)
        x = self.jnp.asarray(x)
        linear_term = self.sufficient_statistics(x)[..., None, :] @ natural_parameters[.., None])[..., 0,0]
        return (self.log_base_measure(x) + linear_term - self.log_partition(natural_parameters))

$\large\text{Consider }N\text{ iid samples }X=\left[x_{1},\dots,x_{N}\right]\text{ from an exponential family distributin }p(x|w))\text{with natural parameters } w\text{ and sufficient statistics }\phi:\mathbb{X}\rightarrow\mathbb{R}^{d}$<br>
$\large p(x_{1},\dots,x_{N}|w)=\Pi_{i=1}^{N}p(x_{i}|w)=exp\left(\left[\sum_{i=1}^{N}\phi(x_{i})\right]^{T}w-Ng(w)+\sum_{i}h(x_{i})\right)$<br>

$\large\text{So to perform inference n the natural parameters }w\text{, we only need to store N and the sufficient statistics}$<br>
$\large\sum_{i=1}^{N}\phi(x_{i}=:N\bar{\phi}_{N}\in\mathbb{R}^{d}$<br>

$\large\text{Therefore for Bayesian inference}$<br>
$\large p(w|X)=\frac{p(X|x)p(x)}{\int p(X|w)p(w)dw}=\frac{\cancel{exp\left[\sum_{i}h(x_{i})\right]}exp(N(\bar{\phi}_{N}w-g(w)))p(w)}{\cancel{exp\left[\sum_{i}h(x_{i})\right]}]\int exp\left(N(\bar{\phi}_{N}w-g(w))\right)p(w)dw}$<br>
$\large\text{Thus computing the likelihood is }\mathcal{O}(N)\text{ and storage is }\mathcal{O}(d)$<br>

$\large\text{Combining Independent Evidence Is Equivalent to Adding Natural Parameters}$<br>
$\large\text{If we have two separate sources of information on an RV }x\text{ with exponential family distributions }p(x|w_{i}), i=1,\dots,M,\text{ then the joint distribution is also an exponential family, whose natual parameters are the sum of the individual natural parameters:}$<br>
$\large p(x, w_{1}, w_{2})\propto p_{w_{1}}(x)p_{w_{2}}(x)=exp\left(\phi(x)^{T}(w_{1}+w_{2})-g(w_{1})-g(w_{2})+2h(x)\right)\propto p_{w_{1}+w_{2}}(x)$<br>
$\large\text{In otw, there is a }\_\_mult\_\_\text{ method for exponential families ( which maps to }\_\_add\_\_\text{ for the natural parameters).}$<br>

<H1>Conjugate Exponential Families can be constructed almost automatically. We just need the log-partition function.</H1>

In [ ]:
# Now we can construct an exponential family almost automatically by providing the log-partition function).
# This code provided in [1] at 54:32 of lecture #13
class ConjugateFamily(ExponentialFamily):
    def __init__(self, likelihood: ExponentialFamily) -> None:
        self._likelihood = likelihood

    def sufficient_statistics(self, w: ArrayLike | jnp.ndarray, /) -> jnp.ndarray: # (D)->(P)
        return jnp.append(w, -self._likelihood.log_partition(w),)
    
    def log_base_measure(self, w: ArrayLike | jnp.ndarray, /) -> jnp.ndarray: # (D)->()
        """The base measure is, implicitly, the Lebesgue measure."""
        w = jnp.asarray(w)
        return jnp.zeros_like(w[..., 0])
    
    def log_partition(self, natural_parameters: ArrayLike | jnp.ndarray, /) -> jnp.ndarray: #(P)->()
        natural_parameters = jnp.asarray(natural_parameters)
        alpha, nu = natural_parameters[:-1], natural_parameters[-1]
        return self._likelihood.conjugate_log_partition(alpha, u)
    
    def unnormalized_logpdf(self, w: ArrayLike | jnp.ndarray, natural_parameters: ArrayLike | jnp.ndarray, /) -> jnp.ndarray: #(d),(P)->()
        """ If the log partition is not available, we can stil compute the unnormalized log pdf."""
        return self.sufficient_statstics(w) @ jnp.asarray(natural_parameters)

<H1>Construct an exponential family almost automatically. We just need the log-partition function</H1>

In [ ]:
# Now we can construct an exponential family almost automatically by providing the log-partition function).
# This code provided in [1] at 52:05 of lecture #13
class ExponentialFamily(abc.ABC):
    def conjugate_log_partition(self, alpha: ArrayLike | jnp.ndarray, nu: ArrayLike | jnp.ndarray, /) -> jnp.ndarray: # (P), ()->()
        """The log partition function of the conjugate exponential family."""
        raise NotImplementedError()
    def conjugate_prior(self) -> "ConjugateFamily":
        return ConjugateFamily(self)
    def posterior_parameters(self, prior_natural_parameters: ArrayLike | jnp.ndarray, data: ArrayLike | jnp.ndarray,) -> jnp.ndarray: # (P),(D)->(P)
        """Computes the natural parameters of the posterior distribution under the conjugate prior. This can be implemented already in the abc and inherited by all sublasses.
        , even if teh conjugate log partition function is not available.
        (In the latter case, only the unnormalized posterior is immediately available, see below)."""
        prior_natural_parameters = jnp.asarray(prior_natural_parameters)
        sufficient_statistics = self.sufficient_statistics(data)
        n = sufficient_statistics.shape[..., 0].size
        expected_sufficient_statistics = jnp.sum(sufficient_statistics, axis=tuple(range(sufficient_statistics.ndim)),)
        alpha_prior, nu_prior = prior_natural_parameters[:-1], prior_natural_parameters[-1]
        return jnp.append(alpha_prior + expected_sufficient_statistics, nu_prior +n)
        


<H1>The expected value of the sufficient statistics can be computed by differentiating g:</H1>


$\large\mathbb{E}_{p(x|w)}[\phi(x)]=\int \phi(x)p(x|w)dx\quad\quad\quad\quad\quad=\int\phi(x)exp[\phi(x)^{T}w-g(w)+h(x)]dx$<br>
$\large\quad\quad\quad\quad\quad\quad=\frac{\int\phi(x)exp[\phi(x)^{T}w+h(x)]dx}{exp(g(w))}\quad\quad\quad\quad\quad=\frac{\int\phi(x)exp[\phi(x)^{T}w+h(x)]dx}{\int exp[\phi(x)^{T}w+h(x)]dx}$<br>
$\large\quad\quad\quad\quad\quad\quad=\frac{\partial log\int exp[\phi(x)^{T}w+h(x)]dx}{\partial w}\quad\quad\quad\quad\quad=\frac{\partial g(w)}{\partial w}$<br>
Differentiation is easy because of autodiff, so for exponential families, we get many integrals (expectations) "for free" as a side product of already knowing the one key integral function $g(w).$[[1, 58:18]](#References)

<H1>Estimating parameters amounts to solving an implicit equation (i.e. an optimization problem).</H1>

$\large\text{Suppose we have }N\text{ iid. samples }X[x_{1},...,]x_{N}]\text{ from teh exponential family }p(x|w)\text{, and we would like to estimate the unknown w.}$<br>
$\large\text{Using }\bar{\phi}_{N}=\sum_{i=1}^{N}\phi(x_{i})\text{ from above, we have}$<br>
$\large log\,p(X|w)=N(\bar{\phi}_{N}^{T}w-g(x))+\sum_{i=1}^{N}h(x_{i})$<br>
$\large\frac{\partial log\,p(X|w)}{\partial w}=N\left(\bar{\phi}_{N}-\frac{\partial g(w)}{\partial w}\right)=0\quad\quad\quad\quad\rightarrow\frac{\partial g(w)}{\partial w}=\bar{\phi}_{N}$<br>
$\large\text{Remember from above that }\frac{}{}=\mathbb{E}_{p(x|w)}[\phi(x)]\text{, so this makes sense.)}$<br>
$\large\text{Even without a conjugate prior (which requires another, p[otentially unknow integral), we can estimate teh parameters of an exponential family from a datset }X\text{ of samples, by solving the implict equation}$<br>
$\large\frac{\partial g(w)}{\partial w}=\bar{\phi}_{N}$<br>
$\large\text{Sometimes this will be analytic, sometimes it will require numerical optimization.}$[[1, 1:00:54]](#References)<br>

<H1>Maximum A Posteriori</H1>

$\large\text{If we first construct the conjugate prior to }p(x|w)\text{, namely}$<br>
$\large p(w|\alpha,\nu)=exp\begin{bmatrix}\begin{pmatrix}w \\ -g(w)\end{pmatrix}&\begin{pmatrix}\alpha&\nu\end{pmatrix}-f(\alpha,\nu)\end{bmatrix}$<br>
$\large\text{then teh MAP estimate from }p(X|w)\text{ is the regularized estimate}$<br>
$\large log\,p(x|X)=N(\bar{\phi}_{N}^{T}w-g(w))+w^{T}\alpha-\nu g(w)+\text{ const.}$<br>
$\large\frac{log\,p(w|X)}{\partial w}=N\bar{\phi}_{N}+\alpha-(N+\nu)frac{\partial g(w)}{\partial w}=0\quad\quad\quad\quad\rightarrow\frac{\partial g(w)}{\partial w}=\frac{N\bar{\phi}_{N}+\alpha}{N+\nu}$<br>
$\large\text{For exponential families, finding teh MAP estimate is just as easy as finding teh MLE. We do not need to know the conjugate prior'[s log-partition function }f(\alpha,\nu)\text{ to find the MAP estimate.}[[1,1:03:54]](#References)

<H1>Conjugate Priors as full Bayesian inference on distributions... if the log partition function is known</H1>

$\large\text{Given a dataset }X=[x_{1},...,x_{N}]\text{ of iid. samples from an exponential family }p(x|w)\text{,}$<br>
$\large\text{then the maximum likelihood estimate is the solution to }\frac{\partial g(w)}{\partial w}=\bar{\phi}_{N}$<br>
$\large\text{the maximum a posteriori estimate is teh solution to }\frac{\partial g(w)}{\partial w}=\frac{N\bar{\phi}_{N}+\alpha}{N+\nu}$<br>
$\large\text{If the conjugate log partition f is known, the full conjugate posterior is }p(x|X)=exp\begin{bmatrix}\begin{pmatrix}w\\-g(w)\end{pmatrix}&\begin{pmatrix}\alpha+\bar{\phi}_{N}\\\nu+N\end{pmatrix}-f(\alpha+\bar{\phi}_{N},\nu+N)\end{bmatrix}$<br>
$\large=p_{\alpha+\bar{\phi}_{N}\,\nu+N}(w)=p_{\alpha_{N},\nu_{N}}(w|X)$<br>
$\large\text{and the predictive for more data }x_{*}\text{ is}$<br>
$\large p(x_{*}|X)=\int p(x_{*}|w)p(x|X)dx=exp[h(x_{*})+f(\alpha_{N}+\phi(x_{*}),\nu_{N}+1)=f(\alpha_{N}, \nu_{N})]$<br>
$\large\text{The algebraic structure of exponential families provides a very powerful scaffold. Bayesian inferwence is "encapsulated" in the log-partition }$<br>
$\large\text{function f of the conjugate prior. But even without it, the conjugate prior is structurally identified, yielding numerical  expressions for MLE and MAP estimation.}$[[1,1:07:24]](#References)<br>

<H1>Tractable estimation</H1>

$\large\text{Exponential families are log-concave, i.e., the log pdf }log p(x|w)\text{ is a concave function of w.}$<br>
$\large\text{Proof: The Hessian of the negative log pdf is teh covariance of the sufficent statistics:}$<br>
$\large -\frac{\partial^{2}log p(\dot|w)}{\partial w_{i}\partial w_{j}}=\frac{\partial^{2} g(w)}{\partial w_{i}\partial w_{j}}=\frac{\partial}{\partial w_{j}}\frac{\partial g(w)}{\partial w_{i}}=\frac{\partial\mathbb{E}_{p(x|w)}[\phi_{i}(x)]}{\partial w_{j}}$<br>
$\large=\frac{\partial}{\partial w_{j}}\int\phi_{i}(x)exp[\phi(x)^{T}w+h(x)-g(w)]dx$<br>
$\large=\int\left[\phi_{i}(x)\phi_{j}(x)-\phi_{i}(x)\frac{\partial g(w)}{\partial w_{j}}\right]exp\left[\phi(x)^{T}w+h(x)-g(w)\right]dx$<br>
$\large=\mathbb{E}_{p(x|w)}[\phi_{i}(x)\phi_{j}(x)]-\mathbb{E}_{p(x|w)}[\phi_{i}(x)]\mathbb{E}_{p(x|w)}[\phi_{j}(x)]$<br>
$\large=cov(\phi_{i}(x),\phi_{j}(x))
$\large\text{This implies that the MLE and MAP estimates are unique and efficiently computable.}$[[1,1:10:42]](#References)<br>


Pick up at 1:14:37 of [[1]](#References)

<a id="References"></a>
<H1>References</H1>
[1.] Probabilistic Machine Learning, Lecture #13, Phillip Hennig, University Tubingen, 3/06/2025, https://www.youtube.com/watch?v=hIauROkubtA&list=PL05umP7R6ij0hPfU7Yuz8J9WXjlb3MFjm&index=13.